In [1]:
import pandas as pd

In [2]:
from homework05 import (
    exercise_1_fama_french, 
    exercise_2_bootstrap_once, 
    exercise_3_bootstrap_pvalues,
    exercise_4_block_bootstrap_pvalues, 
    exercise_5_block_bootstrap_ci
)

**IMPORTANT**

We will use the log excess returns in the following!

In [3]:
df_raw = pd.read_csv("./s5_data.txt", sep="\t")
df_raw.head()

,date,fund1,fund2,fund3,fund4,fund5,fund1_qoq,fund2_qoq,fund3_qoq,fund4_qoq,fund5_qoq,mktrf,smb,hml,rf,mktrf_qoq,smb_qoq,hml_qoq,rf_qoq
0,1990-02-28,1.89,NaN,NaN,-0.18,2.14,-1.645038,NaN,NaN,0.269693,10.951563,1.11,1.03,0.61,0.57,-5.746330,-2.666841,1.779672,1.760223
1,1990-03-31,5.95,NaN,NaN,0.29,-0.16,3.472428,NaN,NaN,0.319708,5.882279,1.83,1.52,-2.89,0.64,-5.122072,1.252816,-1.457389,1.790566
2,1990-04-30,-0.73,NaN,NaN,-0.34,2.07,7.164402,NaN,NaN,-0.230894,4.087491,-3.36,-0.50,-2.55,0.69,-0.499154,2.052828,-4.789039,1.912022
3,1990-05-31,2.00,NaN,NaN,-1.78,0.28,7.280096,NaN,NaN,-1.830078,2.192027,8.42,-2.56,-3.73,0.68,6.694509,-1.573517,-8.896142,2.023490
4,1990-06-30,0.99,NaN,NaN,-0.66,2.45,2.257828,NaN,NaN,-2.759996,4.863513,-1.09,1.43,-1.95,0.63,3.635018,-1.660775,-8.014280,2.013353


## Exercise 1

**ATTENTION**

fund2_qoq contains NaN values!

In [4]:
ff_model = exercise_1_fama_french(df_raw)
ff_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                             OLS Regression Results                             
================================================================================
Dep. Variable:     fund2_qoq_log_excess   R-squared:                       0.691
Model:                              OLS   Adj. R-squared:                  0.687
Method:                   Least Squares   F-statistic:                     157.3
Date:                  Mon, 13 Apr 2026   Prob (F-statistic):           1.49e-53
Time:                          10:05:54   Log-Likelihood:                -580.97
No. Observations:                   215   AIC:                             1170.
Df Residuals:                       211   BIC:                             1183.
Df Model:                             3                                         
Covariance Type:              nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.8393      0.262      3.208      0.002       0.324       1.355
mktrf_qoq      0.6629      0.034     19.357      0.000       0.595       0.730
smb_qoq        0.2231      0.044      5.095      0.000       0.137       0.309
hml_qoq        0.4228      0.043      9.806      0.000       0.338       0.508
==============================================================================
Omnibus:                        9.413   Durbin-Watson:                   0.630
Prob(Omnibus):                  0.009   Jarque-Bera (JB):               10.613
Skew:                           0.381   Prob(JB):                      0.00496
Kurtosis:                       3.777   Cond. No.                         9.09
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

## Exercise 2

In [5]:
ex2_results = exercise_2_bootstrap_once(df_raw)
ex2_results["bootstrap_coefficients"]

const        0.998147
mktrf_qoq    0.673954
smb_qoq      0.200973
hml_qoq      0.462444
dtype: float64

## Exercise 3

We define the null hypothesis as: 

$H_0: \beta_j = 0$

In [6]:
ex3_results = exercise_3_bootstrap_pvalues(df_raw, 10000)
ex3_results["bootstrap_pvalues_two_sided"]

const        0.0004
mktrf_qoq    0.0000
smb_qoq      0.0000
hml_qoq      0.0000
dtype: float64

## Exercise 4

**REMARK**

The homework asks for a **block bootstrap** using `tsboot()` from R’s **boot** package. Since we work in **Python**, we implemented the same idea manually as closely as possible.

Our implementation follows the **moving block bootstrap** logic behind `tsboot(..., sim = "fixed")`:

1. Estimate the original Fama-French regression by OLS.
2. Collect the fitted residual series $\hat{u}_t$.
3. Choose the block length as
   $$
   l = \lfloor T^{1/3} \rfloor,
   $$
   in line with the homework and TA script.
4. Construct **overlapping blocks** of consecutive residuals:
   $$
   (\hat{u}_1, \ldots, \hat{u}_l), \quad
   (\hat{u}_2, \ldots, \hat{u}_{l+1}), \quad \ldots
   $$
5. Resample these blocks **with replacement** until a bootstrap residual series $\hat{u}_t^*$ of length $T$ is obtained.
6. Form the artificial response
   $$
   y_t^* = \hat{y}_t + \hat{u}_t^*
   $$
   and re-estimate the regression.
7. Repeat this many times to obtain the bootstrap distribution of the coefficients.

This reproduces the main purpose of `tsboot()`: unlike standard residual resampling, it preserves the **local time dependence** and **autocorrelation structure** in the residuals by resampling **blocks of consecutive observations** rather than individual residuals.

In [7]:
ex4_results = exercise_4_block_bootstrap_pvalues(df_raw)
ex4_results["bootstrap_pvalues_two_sided"]

const        0.0310
mktrf_qoq    0.0000
smb_qoq      0.0002
hml_qoq      0.0000
dtype: float64

### Explanation Exercise 4

The block bootstrap preserves the serial dependence in the residuals, whereas the standard bootstrap resamples individual residuals as if they were iid and therefore breaks that dependence structure. Because positive autocorrelation reduces the effective amount of independent information, it increases the uncertainty of the coefficient estimates. As a result, the block bootstrap typically produces a wider bootstrap distribution and therefore higher p-values.

## Exercise 5

In [8]:
ex5_results = exercise_5_block_bootstrap_ci(df_raw)
ex5_results["confidence_intervals"]

,ci_lower,ci_upper
const,0.195154,1.601789
mktrf_qoq,0.584978,0.740586
smb_qoq,0.129995,0.315752
hml_qoq,0.322452,0.523468
